# Statistical Analysis — AG2 Stage 2 Interventions

Paired statistical tests comparing each Stage 2 intervention against the Stage 1 baseline on the same OlympiadBench tasks.

| Stage | Target FM | Prompt addition |
|-------|-----------|----------------|
| **v1** | FM-2.6 Action-Reasoning Mismatch | Self-contained code blocks; explicitly match description |
| **v2** | FM-1.1 Disobey Task Specification | Provide general solution; exact output format |
| **v3** | FM-3.3 Weak Verification | Verify answer by back-substitution before TERMINATE |

**Statistical methods:**
- **McNemar test** (paired binary, applied to accuracy and every FM): exact binomial when b+c < 25; χ² with continuity correction otherwise
- **Bootstrap 95% CI**: 10 000 paired resamples keyed on `question_id`, seed = 42

★ marks the target FM for each intervention.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
from IPython.display import display

## Configuration

In [ ]:
RESULTS = Path('results')

BASELINE = RESULTS / 'baseline_olympiad_gpt41_n50_20260609'

INTERVENTIONS = [
    (
        'v1 – FM-2.6 Action-Reasoning Mismatch',
        RESULTS / 'stage_2_v1_olympiad_gpt41_n50_20260611',
        '2.6',
    ),
    (
        'v2 – FM-1.1 Disobey Task Specification',
        RESULTS / 'stage_2_v2_olympiad_gpt41_n50_20260612',
        '1.1',
    ),
    (
        'v3 – FM-3.3 Weak Verification',
        RESULTS / 'stage_2_v3_olympiad_gpt41_n50_20260613',
        '3.3',
    ),
]

FM_CODES = [
    '1.1', '1.2', '1.3', '1.4', '1.5',
    '2.1', '2.2', '2.3', '2.4', '2.5', '2.6',
    '3.1', '3.2', '3.3',
]

FM_NAMES = {
    '1.1': 'Disobey Task Specification',
    '1.2': 'Disobey Role Specification',
    '1.3': 'Step Repetition',
    '1.4': 'Loss of Conversation History',
    '1.5': 'Unaware of Termination Conditions',
    '2.1': 'Conversation Reset',
    '2.2': 'Fail to Ask for Clarification',
    '2.3': 'Task Derailment',
    '2.4': 'Information Withholding',
    '2.5': "Ignored Other Agent's Input",
    '2.6': 'Action-Reasoning Mismatch',
    '3.1': 'Premature Termination',
    '3.2': 'No or Incorrect Verification',
    '3.3': 'Weak Verification',
}

## Data Loading

In [4]:
def load_run(run_dir: Path, label: str) -> pd.DataFrame:
    pred = pd.read_csv(run_dir / 'saved_results_1shot' / 'predictions.csv')
    summ = pd.read_csv(run_dir / 'summary.csv')[['trace_id', 'question_id', 'correct']]
    # Robustly parse correct column regardless of bool/string encoding
    summ['correct'] = (
        summ['correct']
        .map({'True': True, 'False': False, True: True, False: False})
        .astype(bool)
    )
    df = pred.merge(summ, on='trace_id', how='left')
    df['_run'] = label
    return df


df_base = load_run(BASELINE, 'Baseline')
print(f'Baseline : {len(df_base):3d} traces  |  accuracy = {df_base["correct"].mean():.3f}')
print()
for label, idir, fm in INTERVENTIONS:
    df_i = load_run(idir, label)
    print(f'{label}')
    print(f'  {len(df_i):3d} traces  |  accuracy = {df_i["correct"].mean():.3f}  |  target FM-{fm} prevalence = {df_i[fm].mean():.3f}')

Baseline :  49 traces  |  accuracy = 0.653

v1 – FM-2.6 Action-Reasoning Mismatch
   49 traces  |  accuracy = 0.633  |  target FM-2.6 prevalence = 0.265
v2 – FM-1.1 Disobey Task Specification
   48 traces  |  accuracy = 0.708  |  target FM-1.1 prevalence = 0.125


FileNotFoundError: [Errno 2] No such file or directory: 'results\\stage_2_v3_olympiad_gpt41_n50_20260612\\saved_results_1shot\\predictions.csv'

## Pairing & Statistical Functions

In [ ]:
def pair_runs(df_left: pd.DataFrame, df_right: pd.DataFrame) -> pd.DataFrame:
    """Inner join on question_id → one row per shared task."""
    left  = df_left.set_index('question_id')
    right = df_right.set_index('question_id')
    paired = left.merge(
        right, left_index=True, right_index=True, suffixes=('_base', '_int')
    )
    return paired.reset_index()

In [ ]:
def mcnemar_test(base_bin: np.ndarray, int_bin: np.ndarray):
    """
    McNemar test for paired binary outcomes.

    For accuracy:  base=1 means baseline correct,  int=1 means intervention correct.
    For FM columns: base=1 means FM present in baseline, int=1 means FM present in intervention.

    Returns (b, c, statistic, p_value, method):
      b = base=1 & int=0
      c = base=0 & int=1
    Uses exact binomial when b+c < 25, chi-square with continuity correction otherwise.
    """
    x = base_bin.astype(int)
    y = int_bin.astype(int)
    b = int(((x == 1) & (y == 0)).sum())
    c = int(((x == 0) & (y == 1)).sum())
    n = b + c
    if n == 0:
        return b, c, np.nan, np.nan, '—'
    if n < 25:
        p = stats.binomtest(min(b, c), n=n, p=0.5, alternative='two-sided').pvalue
        return b, c, float(min(b, c)), float(p), 'exact binomial'
    stat = (abs(b - c) - 1) ** 2 / n
    p = stats.chi2.sf(stat, df=1)
    return b, c, float(stat), float(p), 'χ² (CC)'


def bootstrap_ci(
    base_vals: np.ndarray,
    int_vals: np.ndarray,
    n_boot: int = 10_000,
    seed: int = 42,
):
    """Paired bootstrap 95% CI on mean(intervention) − mean(baseline)."""
    n   = len(base_vals)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_diffs = int_vals[idx].mean(axis=1) - base_vals[idx].mean(axis=1)
    return float(np.percentile(boot_diffs, 2.5)), float(np.percentile(boot_diffs, 97.5))

## Table Building & Display Helpers

In [ ]:
def _fmt_p(p: float) -> str:
    if np.isnan(p):
        return '—'
    if p < 0.001:
        return '< 0.001'
    return f'{p:.3f}'


def _fmt_ci(lo: float, hi: float) -> str:
    def _s(v):
        return f'+{v:.3f}' if v >= 0 else f'{v:.3f}'
    return f'[{_s(lo)}, {_s(hi)}]'


def _fmt_diff(d: float) -> str:
    return f'+{d:.3f}' if d >= 0 else f'{d:.3f}'


def build_summary_table(int_dir: Path, target_fm: str) -> pd.DataFrame:
    """Return a 15-row summary DataFrame for one intervention vs baseline."""
    df_int = load_run(int_dir, 'Intervention')
    paired = pair_runs(df_base, df_int)
    n_paired = len(paired)

    rows = []

    # ── Accuracy ─────────────────────────────────────────────────────────────
    bv = paired['correct_base'].astype(float).values
    iv = paired['correct_int'].astype(float).values
    b, c, stat, p, method = mcnemar_test(bv, iv)
    ci_lo, ci_hi = bootstrap_ci(bv, iv)
    rows.append({
        'Metric'       : 'Accuracy',
        'Baseline'     : round(float(bv.mean()), 3),
        'Intervention' : round(float(iv.mean()), 3),
        'Difference'   : _fmt_diff(iv.mean() - bv.mean()),
        '95% CI'       : _fmt_ci(ci_lo, ci_hi),
        'p-value'      : _fmt_p(p),
        'Method'       : method,
        '_target'      : False,
        '_n_paired'    : n_paired,
    })

    # ── FM rows ───────────────────────────────────────────────────────────────
    for fm in FM_CODES:
        bv = paired[f'{fm}_base'].astype(float).values
        iv = paired[f'{fm}_int'].astype(float).values
        b, c, stat, p, method = mcnemar_test(bv, iv)
        ci_lo, ci_hi = bootstrap_ci(bv, iv)
        star = ' ★' if fm == target_fm else ''
        rows.append({
            'Metric'       : f'FM {fm}  {FM_NAMES[fm]}{star}',
            'Baseline'     : round(float(bv.mean()), 3),
            'Intervention' : round(float(iv.mean()), 3),
            'Difference'   : _fmt_diff(iv.mean() - bv.mean()),
            '95% CI'       : _fmt_ci(ci_lo, ci_hi),
            'p-value'      : _fmt_p(p),
            'Method'       : method,
            '_target'      : fm == target_fm,
            '_n_paired'    : n_paired,
        })

    return pd.DataFrame(rows)


def display_summary(df: pd.DataFrame, title: str) -> None:
    n = int(df['_n_paired'].iloc[0])
    vis_cols = ['Metric', 'Baseline', 'Intervention', 'Difference', '95% CI', 'p-value']

    def _highlight(row):
        if row['_target']:
            return ['background-color: #fff9c4; font-weight: bold'] * len(row)
        return [''] * len(row)

    styled = (
        df[vis_cols + ['_target']]
        .style
        .apply(_highlight, axis=1)
        .hide(axis='index')
        .hide(['_target'], axis='columns')
        .set_caption(f'{title}  (n = {n} paired tasks)')
        .set_properties(
            **{'text-align': 'right'},
            subset=['Baseline', 'Intervention', 'Difference', '95% CI', 'p-value'],
        )
        .set_properties(**{'text-align': 'left'}, subset=['Metric'])
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
            {'selector': 'th',
             'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
        ])
    )
    display(styled)

---
## Intervention v1 — FM-2.6 Action-Reasoning Mismatch

**Prompt addition:** *"Each code block must be completely self-contained: always include all necessary imports and variable definitions, even if they appeared in a previous block. Before writing code, explicitly state what the code will compute and ensure the implementation matches that description exactly."*

★ marks the target failure mode.

In [ ]:
v1_label, v1_dir, v1_fm = INTERVENTIONS[0]
t_v1 = build_summary_table(v1_dir, v1_fm)
display_summary(t_v1, v1_label)

---
## Intervention v2 — FM-1.1 Disobey Task Specification

**Prompt addition:** *"Always provide the general solution or expression requested by the task, even if a specific numerical example is used to verify the approach. Final answers must follow the requested output format exactly. Use exact symbolic forms such as fractions or radicals instead of decimal approximations unless the problem explicitly asks for decimals."*

★ marks the target failure mode.

In [ ]:
v2_label, v2_dir, v2_fm = INTERVENTIONS[1]
t_v2 = build_summary_table(v2_dir, v2_fm)
display_summary(t_v2, v2_label)

---
## Intervention v3 — FM-3.3 Weak Verification

**Prompt addition:** *"After obtaining a result from code, verify the answer by substituting it back into the original problem or checking it against a known constraint before writing TERMINATE."*

★ marks the target failure mode.

In [ ]:
v3_label, v3_dir, v3_fm = INTERVENTIONS[2]
t_v3 = build_summary_table(v3_dir, v3_fm)
display_summary(t_v3, v3_label)

---
## Combined Overview — All Three Interventions

Side-by-side comparison of Δ (intervention − baseline, in proportion units) and McNemar p-value across all three interventions.
Highlighted cells mark the target FM for each intervention.

In [ ]:
def build_combined(table_triples):
    """
    table_triples: list of (label, target_fm, df) from build_summary_table.
    Aligns rows by position (all tables share the same 15-row structure).
    """
    # Clean metric names (strip star markers) from the first table
    metrics = table_triples[0][2]['Metric'].str.replace(' ★', '', regex=False).tolist()

    combined = pd.DataFrame({'Metric': metrics})
    target_col_pairs = {}  # fm_code -> (delta_col_idx, p_col_idx)

    for label, target_fm, df in table_triples:
        short = label.split('–')[0].strip()       # 'v1', 'v2', 'v3'
        delta_col = f'{short} Δ'
        p_col     = f'{short} p'
        combined[delta_col] = df['Difference'].tolist()
        combined[p_col]     = df['p-value'].tolist()
        target_col_pairs[target_fm] = (delta_col, p_col)

    return combined, target_col_pairs


combined, target_cols = build_combined([
    (v1_label, v1_fm, t_v1),
    (v2_label, v2_fm, t_v2),
    (v3_label, v3_fm, t_v3),
])

# Map each target FM to its row index in the combined table
target_row_idx = {
    fm: combined.index[combined['Metric'].str.contains(fm, regex=False)].tolist()[0]
    for fm in [v1_fm, v2_fm, v3_fm]
}

# Column index lookup for cell-level styling
col_list = combined.columns.tolist()

def _hl_combined(row):
    styles = [''] * len(row)
    ri = row.name
    for fm, (dcol, pcol) in target_cols.items():
        if ri == target_row_idx[fm]:
            for col in [dcol, pcol]:
                ci = col_list.index(col)
                styles[ci] = 'background-color: #fff9c4; font-weight: bold'
    return styles


styled_combined = (
    combined.style
    .apply(_hl_combined, axis=1)
    .hide(axis='index')
    .set_caption('Combined Overview — Δ and p-value for all three interventions vs baseline')
    .set_properties(
        **{'text-align': 'right'},
        subset=combined.columns[1:].tolist(),
    )
    .set_properties(**{'text-align': 'left'}, subset=['Metric'])
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '1.05em'), ('font-weight', 'bold'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('text-align', 'center'), ('background-color', '#f5f5f5')]},
    ])
)
display(styled_combined)

---
## Notes

**McNemar test interpretation:**
- For *accuracy* rows: b = tasks where baseline was correct and intervention was wrong; c = tasks where baseline was wrong and intervention was correct.
- For *FM* rows: b = tasks where the FM was present in baseline but absent in intervention; c = tasks where FM was absent in baseline but present in intervention.
- A significant p-value on a FM row means the intervention reliably changed FM prevalence on those specific tasks (not just overall rate differences).

**Bootstrap CI interpretation:**
- A 95% CI that excludes zero indicates the observed difference is unlikely to be due to sampling variation at the α = 0.05 level.
- The CI is directly interpretable as the plausible range of true proportion differences.

**Why paired tests matter:**
- Different runs completed slightly different subsets (baseline = 49, v1 = 49, v2 = 48, v3 = 50 traces). The inner join on `question_id` ensures every comparison uses exactly the same tasks.
- Sample sizes are small (≈48), so exact binomial is used for most McNemar tests (b+c typically < 25); interpret p-values as approximate guides rather than strict thresholds.